# Building and Running a Model Locally with Podman

Before deploying a model to the Datamint server, it's worth running it as a real container on your
own machine first, a "local deploy". It's the same image format and the same `/invocations`
serving contract the server uses, just built and run entirely on your machine with `podman`. If it
builds and predicts correctly here, you've already caught most of what would go wrong on a real
deployment.

`02_deploy_registered_model.ipynb` covers the next step: taking a model you've validated locally
and actually deploying it on the Datamint server.

> **IMPORTANT:** This notebook requires `podman` to be installed **and running** on your machine,
> the Python `podman` package alone is not enough, it just talks to the podman API socket.
>
> ```bash
> # rootless (typical for local dev)
> systemctl --user enable --now podman.socket
> ```
>

## Setup

In [ ]:
import datamint.mlflow as datamint_mlflow
from datamint import Api

PROJECT_NAME = 'Segmentation_Tutorial'
MODEL_NAME   = 'Segmentation_Tutorial_Model'

api = Api()
datamint_mlflow.set_project(PROJECT_NAME)

## 1. Pick a Model Version and a Resource


In [ ]:
model_uri = f'models:/{MODEL_NAME}/latest'

resources = list(api.resources.get_list(project_name=PROJECT_NAME, limit=1))
resource = resources[0]

print('Model URI:', model_uri)
print('Resource :', resource.filename, resource.id)

## 2. Build the Podman Image

`build_docker_image` downloads the model, generates a serving `Dockerfile`, and runs `podman build`, all locally. Build output streams to stdout as it happens, pass `log_callback` if you'd rather
capture it yourself (e.g. into a file).

`with_gpu=False` here since this is a CPU-only walkthrough.

In [ ]:
from datamint.mlflow.models.docker_build import (
    build_docker_image,
    remove_docker_image,
    run_docker_container,
    stop_docker_container,
)

built = build_docker_image(model_uri, with_gpu=False)
print(built)

## 3. Run the Container

Starts the image with `podman run`, publishing the model server's port to the host. By default
`run_docker_container` polls the container's `/ping` endpoint until it responds (or raises if it
crashes/times out first), so by the time this cell returns the container is ready for requests.

In [ ]:
running = run_docker_container(built.image_name, built.image_tag)
print(running)

## 4. Predict Against the Running Container

`predict_local` posts directly to the container's `/invocations` endpoint. Pass `resource_id` + `api_client` to run on a resource
already in Datamint (as below), or `file_path` for a local file instead.

In [ ]:
from datamint.mlflow.models.local_inference import predict_local

annotations = predict_local(
    container_port=running.host_port,
    resource_id=resource.id,
    api_client=api,
)

print(f'Annotations: {len(annotations)}')
for ann in annotations:
    print(' ', ann)

## 5. Clean Up

These containers/images are meant to be disposable dev artifacts, not left running. `stop_docker_container`
stops (and by default removes) the container; `remove_docker_image` drops the built image too.

In [ ]:
stop_docker_container(running)
remove_docker_image(built.image_name, built.image_tag)

## Summary

This notebook covered:
- Building a podman image for a registered model entirely on your own machine
- Running that image locally and waiting for it to become ready
- Predicting against it, with results parsed into `Annotation` objects
- Tearing the container and image back down when you're done

Model behaving as expected here means it's ready for the next step: deploying it for real on the
Datamint server, covered in `02_deploy_registered_model.ipynb`.